# Consistent comparison: Ani and DBSCAN recipes

Eight repeats of five-fold CV on all 235 verified labels. Compare the two original recipes on identical held-out trials and use shared feature-access controls to distinguish pipeline differences. All settings, policies, and seeds are fixed before scoring. No Kaggle submission or new parameter search.

**Current policy for future work:** use only the saved training-only branches and folds (Ani sample F1 0.908865; DBSCAN 0.888316). This notebook records the historical four-variant experiment. Its transductive branches are retained for reproducibility and are not the default for new candidate runs. See experiments/paired_recipe_cv/active_benchmark.json and BENCHMARK.md.

## Data, frozen recipes, and protocol

In [1]:
from pathlib import Path
import ast
import hashlib
import importlib.metadata
import json
import platform
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, normalize
from threadpoolctl import threadpool_limits

REASONS = ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']
FEATURES = [f'emb_{i}' for i in range(150)]
SEEDS = list(range(100, 108))
N_FOLDS = 5
VARIANTS = ['dbscan_train_only', 'ani_transductive', 'ani_train_only', 'dbscan_transductive']
COMPARISONS = {
    'native_recipes': ('dbscan_train_only', 'ani_transductive'),
    'shared_train_only': ('dbscan_train_only', 'ani_train_only'),
    'shared_transductive': ('dbscan_transductive', 'ani_transductive'),
}
LABEL_RULE = dict(minimum_support=3., prior_strength=2., positive_threshold=0.6,
                  negative_threshold=0.1, maximum_generated_weight=0.25, maximum_class_fraction=0.5)
cwd = Path.cwd()
DATA = next(p for p in [cwd, cwd/'part1', cwd/'AML-Midterm'/'part1']
            if (p/'part1_train.csv').is_file() or (p/'data/part1_train.csv').is_file()).resolve()
INPUT = DATA if (DATA/'part1_train.csv').is_file() else DATA/'data'
OUT = DATA/'experiments/paired_recipe_cv'
for folder in ['splits', 'preprocessing', 'models', 'datasets', 'predictions', 'controls']:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT/'part1_train.csv')
valid = pd.read_csv(INPUT/'part1_val.csv')
test = pd.read_csv(INPUT/'part1_test.csv')
all_frame = pd.concat([train[['ID']+FEATURES], valid[['ID']+FEATURES], test[['ID']+FEATURES]], ignore_index=True)
X = all_frame[FEATURES].to_numpy()
Y = np.full((len(X), 5), -100, dtype=int)
Y[:len(train)] = train[REASONS].to_numpy(dtype=int)
Y[len(train):len(train)+len(valid)] = valid[REASONS].to_numpy(dtype=int)
verified = np.isin(Y, [0, 1]).all(axis=1)
known_rows = np.flatnonzero(verified)
original_known = np.flatnonzero(verified[:len(train)])
original_unlabeled = np.flatnonzero(~verified[:len(train)])
valid_rows = np.arange(len(train), len(train)+len(valid))
test_rows = np.arange(len(train)+len(valid), len(X))
ALL_ROWS = np.arange(len(X))
source_group = np.array(['original_training']*len(train)+['former_validation']*len(valid)+['test']*len(test))
assert len(X) == 1561 and len(known_rows) == 235 and len(original_known) == 157
assert len(original_unlabeled) == 1014 and len(valid_rows) == 78 and len(test_rows) == 312
assert all_frame.ID.is_unique and np.isfinite(X).all() and (Y[~verified] == -100).all()
SOURCE = DATA/'experiments/additional_methods'
OLD_MANIFEST = json.loads((SOURCE/'submission_manifest.json').read_text())
OLD_MODEL = joblib.load(SOURCE/OLD_MANIFEST['model_file'])
ANI_COMPONENT_PATH = DATA/'experiments/ani_ensemble/models/ani_validation_component.joblib'
ANI_COMPONENT = joblib.load(ANI_COMPONENT_PATH)
ANI_SOURCE = DATA/'experiments/ani_ensemble/ani_source/part1_best_model.ipynb'
assert hashlib.sha256(ANI_SOURCE.read_bytes()).hexdigest() == '06ac000e36d820d9e8c3e96c21b07631e6716e4f29883e726a8fcfc64c1ddbe7'
protected_paths = [INPUT/f for f in ['part1_train.csv', 'part1_val.csv', 'part1_test.csv']]
protected_paths += [SOURCE/OLD_MANIFEST['model_file'], SOURCE/OLD_MANIFEST['submission_file'],
                    SOURCE/'datasets/dbscan_m5_q25.csv', SOURCE/'submission_manifest.json',
                    ANI_SOURCE, ANI_COMPONENT_PATH,
                    DATA/'experiments/final_dbscan_trainval235/models/dbscan_logistic_trainval235.joblib']
PROTECTED = {str(p.relative_to(DATA)): hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
dataset = all_frame.copy()
dataset['source_group'] = source_group
for j, reason in enumerate(REASONS):
    dataset[reason] = Y[:, j]
dataset.to_csv(OUT/'datasets/shared_raw_features_labels.csv', index=False)

PROTOCOL = dict(
    label_pool=235, folds=5, repeats=8, seeds=SEEDS, splitter='sklearn.model_selection.KFold(shuffle=True)',
    split_reason='Match Ani original CV fold count, seeds, and ordering; same assignments for every variant',
    fitting_labels_per_fold=188, held_labels_per_fold=47,
    primary_comparison='native_recipes', comparisons=COMPARISONS, variants=VARIANTS,
    train_only_policy='Fit scalers/clustering to remaining 188 verified + 1014 unlabeled training rows; no held-out or test features enter fitting; pseudo-label only 1014 unlabeled training rows',
    transductive_policy='Fit feature-only scaler/clustering to all 1561 embeddings; label-dependent fitting uses 188 verified labels and may pseudo-label the same 1014 originally unlabeled + 312 test rows; held-out known rows never receive pseudo-labels',
    classifier_scaling='DBSCAN classifier scaler uses verified fitting rows only; Ani scaler uses the permitted feature pool; preserve recipe-specific choices',
    ani_recipe=dict(C=.3, knn=10, knn_weight=.3, pca=30, gmm_components=30, covariance='diag',
                    gmm_weight=.2, confidence=.8, rounds=2, cutoff=.3, random_state=0),
    dbscan_recipe=dict(C=1000., tol=1e-4, solver='lbfgs', max_iter=5000, min_samples=5,
                       eps_quantile=.25, eps_floor=1e-6, cutoff=.5, label_rule=LABEL_RULE),
    primary_metric='sample F1 pooled over all 235 out-of-fold predictions within repeat, then mean of eight repeats',
    secondary_metrics='macro/per-reason F1, paired fold/repeat differences, original-training/former-validation subgroups',
    no_tuning=True, no_protocol_selection_after_scores=True, no_test_labels=True,
    interpretation='Known Kaggle scores motivate the audit but are not used to choose seeds, settings, data policy, or winning protocol; agreement across two recipes does not independently validate a model-selection system',
    known_kaggle_scores=dict(ani_235=.89423, dbscan_157=.875, dbscan_235=.86452),
    protected_input_sha256=PROTECTED)
(OUT/'protocol.json').write_text(json.dumps(PROTOCOL, indent=2)+'\n')
(OUT/'PROTOCOL.md').write_text(
    '# Frozen paired recipe evaluation\n\n'
    'Before fitting: eight repeats of five-fold CV over all 235 verified labels, KFold seeds 100–107 '
    'and original training-known then validation ordering, matching Ani’s original CV. Each fit sees '
    '188 verified labels and scores the same 47 held-out labels for every variant. No parameter search.\n\n'
    'Primary: compare the two original recipes with their native feature-access policies '
    '(DBSCAN training-only, Ani transductive). Controls: compare both with training-only features, '
    'and both with transductive feature access. These controls modify feature access deliberately; '
    'they are not previously submitted models. Preserve each classifier’s own scaling, training rules, '
    'and fixed prediction cutoff.\n\n'
    'Feature-only transductive preprocessing may see held-out embeddings, never held-out labels. '
    'Held-out known rows are excluded from classifier/label-mapping targets and from every pseudo-label '
    'training pool in both policies. Test labels are never available. Each round rebuilds label mapping '
    'from permitted fitting labels. Train-only preprocessing is refitted separately inside each fold.\n\n'
    'Primary metric: pooled trial sample F1 within repeat, then the eight-repeat mean. Macro/per-label '
    'scores and paired fold/repeat differences are secondary. Repeats share trials; dispersion is descriptive, '
    'not a confidence interval from independent tests. No protocol will be selected based on whether '
    'its ranking resembles the already-known Kaggle ranking. No new submission or full-data refit.\n')
print('Protocol frozen before scores: 235 labels; 8 x 5 identical folds; four fixed variants; three predeclared paired comparisons.')



Protocol frozen before scores: 235 labels; 8 x 5 identical folds; four fixed variants; three predeclared paired comparisons.


## Fitting helpers with explicit feature/label access

In [2]:
# The following two base-model helpers reproduce Ani’s stored implementation.
def fit_ani_base(Xl, Yl):
    rows=[]; ys=[]; ws=[]
    for i,y in enumerate(Yl):
        positive=np.flatnonzero(y)
        rows.extend([i]*len(positive)); ys.extend(positive.tolist()); ws.extend([1./len(positive)]*len(positive))
    rows=np.asarray(rows); y=np.asarray(ys); weights=np.asarray(ws)
    counts=np.bincount(y,weights=weights,minlength=5)+1e-9
    weights=weights*counts.mean()/counts[y]
    estimator=LogisticRegression(C=.3,max_iter=3000)
    with warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        estimator.fit(Xl[rows],y,sample_weight=weights)
    return dict(lr=estimator, neighbor_X_normalized=normalize(Xl), neighbor_targets=Yl/Yl.sum(axis=1,keepdims=True),
                neighbor_count=10, knn_weight=.3)


def predict_ani_base(base,Xq):
    lr_prob=np.zeros((len(Xq),5)); lr_prob[:,base['lr'].classes_]=base['lr'].predict_proba(Xq)
    similarities=normalize(Xq)@base['neighbor_X_normalized'].T
    indices=np.argsort(-similarities,axis=1)[:,:base['neighbor_count']]
    weights=np.take_along_axis(similarities,indices,axis=1).clip(0)**2
    knn_prob=(weights[:,:,None]*base['neighbor_targets'][indices]).sum(axis=1)/(weights.sum(axis=1,keepdims=True)+1e-9)
    return (1-base['knn_weight'])*lr_prob+base['knn_weight']*knn_prob


def decide(probability, cutoff):
    assert np.isfinite(probability).all() and ((probability>=0)&(probability<=1+1e-12)).all()
    predicted=(probability>=cutoff).astype(int)
    empty=predicted.sum(axis=1)==0; indices=np.flatnonzero(empty)
    predicted[indices,probability[indices].argmax(axis=1)]=1
    return predicted,empty


def metrics(actual, probability, cutoff):
    predicted,empty=decide(probability,cutoff)
    return dict(sample_f1=float(f1_score(actual,predicted,average='samples',zero_division=0)),
                macro_f1=float(f1_score(actual,predicted,average='macro',zero_division=0)),
                fallback_trials=int(empty.sum()),
                **dict(zip([f'f1_{r}' for r in REASONS],f1_score(actual,predicted,average=None,zero_division=0).tolist())))


def predict_packet(packet, raw):
    transformed=packet['scaler'].transform(np.asarray(raw))
    if packet['family']=='ani':
        return predict_ani_base(packet['base'],transformed)
    return np.column_stack([est.predict_proba(transformed)[:,list(est.classes_).index(1)] for est in packet['estimators']])


def prepare_features(name, feature_rows):
    scaler=StandardScaler().fit(X[feature_rows])
    scaled=scaler.transform(X)
    fitting=scaled[feature_rows]
    with threadpool_limits(limits=1), warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        nearest=NearestNeighbors(n_neighbors=5,algorithm='brute',n_jobs=1).fit(fitting).kneighbors(fitting)[0][:,-1]
        eps=max(float(np.quantile(nearest,.25)),1e-6)
        dbscan=DBSCAN(eps=eps,min_samples=5,metric='euclidean',algorithm='brute',n_jobs=1).fit(fitting)
        pca=PCA(30,random_state=0).fit(fitting)
        gmm=GaussianMixture(30,covariance_type='diag',n_init=3,random_state=0).fit(pca.transform(fitting))
        assert gmm.converged_
    db_ids=np.full(len(X),-2,dtype=int); db_ids[feature_rows]=dbscan.labels_
    responsibility=gmm.predict_proba(pca.transform(scaled))
    packet=dict(name=name,scaler=scaler,pca=pca,gmm=gmm,dbscan=dbscan,eps=eps,
                feature_rows=feature_rows,db_ids=db_ids,ani_responsibility=responsibility)
    joblib.dump(packet,OUT/'preprocessing'/f'{name}.joblib',compress=3)
    return packet


def pseudo_targets(pool, fitting_rows, preprocessor):
    assert set(fitting_rows).issubset(pool)
    local_verified=np.isin(pool,fitting_rows)
    labels=np.full((len(pool),5),-100,dtype=int)
    labels[local_verified]=Y[pool[local_verified]]
    ids=preprocessor['db_ids'][pool]
    assert (ids>=-1).all()
    components=np.unique(ids[ids>=0])
    responsibility=np.column_stack([ids==c for c in components]).astype(float) if len(components) else np.empty((len(pool),0))
    support_component=responsibility[local_verified].sum(axis=0)
    prior=labels[local_verified].mean(axis=0)
    positive_component=responsibility[local_verified].T@labels[local_verified]
    rates=(positive_component+2*prior)/(support_component[:,None]+2)
    probability=responsibility@rates; support=responsibility@support_component
    target=np.full(labels.shape,-100,dtype=int); weight=np.zeros(labels.shape)
    target[local_verified]=labels[local_verified];weight[local_verified]=1.
    eligible=(~local_verified)&(support>=3)
    for j in range(5):
        positive=eligible&(probability[:,j]>=.6); negative=eligible&(probability[:,j]<=.1)
        np_,nn=int(positive.sum()),int(negative.sum())
        kp=int(labels[local_verified,j].sum());kn=int(local_verified.sum()-kp)
        if not np_ or not nn:continue
        fraction=min(.5,.25*np_/kp,.25*nn/kn)
        target[positive,j]=1;target[negative,j]=0
        weight[positive,j]=fraction*kp/np_;weight[negative,j]=fraction*kn/nn
        assert weight[~local_verified,j].max()<=.25+1e-12
        np.testing.assert_allclose(weight[positive,j].sum()/kp,weight[negative,j].sum()/kn,atol=1e-12)
    assert np.array_equal(weight>0,target!=-100)
    assert (weight[(ids<0)&~local_verified]==0).all()
    return target,weight,local_verified,probability,support


FOLD_RESULTS=[];FIT_AUDIT=[];GENERATION_AUDIT=[];ROLE_AUDIT=[]

def fit_dbscan(name, policy, fitting_rows, held_rows, preprocessor, persist=True):
    unlabeled=original_unlabeled if policy=='train_only' else np.r_[original_unlabeled,test_rows]
    pool=np.sort(np.r_[fitting_rows,unlabeled])
    assert not len(np.intersect1d(pool,held_rows))
    target,weight,is_verified,probability,support=pseudo_targets(pool,fitting_rows,preprocessor)
    scaler=StandardScaler().fit(X[fitting_rows]); scaled=scaler.transform(X[pool])
    normalized_weight=weight*len(fitting_rows)/weight.sum(axis=0)
    estimators=[]
    for j,reason in enumerate(REASONS):
        included=weight[:,j]>0;estimator=clone(OLD_MODEL['estimators'][j])
        with warnings.catch_warnings():
            warnings.simplefilter('error',ConvergenceWarning)
            estimator.fit(scaled[included],target[included,j],sample_weight=normalized_weight[included,j])
        estimators.append(estimator)
        if persist:
            FIT_AUDIT.append(dict(context=name,variant='dbscan_'+policy,reason=reason,round='final',
                                  fitting_rows=int(included.sum()),verified_rows=len(fitting_rows),iterations=int(estimator.n_iter_.max())))
            GENERATION_AUDIT.append(dict(context=name,variant='dbscan_'+policy,reason=reason,
                                         generated_positive=int(((target[:,j]==1)&~is_verified).sum()),
                                         generated_negative=int(((target[:,j]==0)&~is_verified).sum())))
    packet=dict(family='dbscan',policy=policy,scaler=scaler,estimators=estimators,native_cutoff=.5,
                features=FEATURES,reasons=REASONS,fitting_rows=fitting_rows,pool_rows=pool,
                preprocessing_file=f'preprocessing/{preprocessor["name"]}.joblib')
    frame=all_frame.iloc[pool][['ID']].copy();frame['global_row']=pool;frame['verified_fit']=is_verified
    frame['cluster']=preprocessor['db_ids'][pool];frame['known_support']=support
    for j,reason in enumerate(REASONS):
        frame[reason]=target[:,j];frame[f'weight_{reason}']=weight[:,j]
        frame[f'normalized_weight_{reason}']=normalized_weight[:,j]
    if persist:
        frame.to_csv(OUT/'datasets'/f'{name}__dbscan_{policy}.csv',index=False)
    return packet,frame


def fit_ani(name,policy,fitting_rows,held_rows,preprocessor,persist=True):
    unlabeled=original_unlabeled if policy=='train_only' else np.r_[original_unlabeled,test_rows]
    assert not len(np.intersect1d(np.r_[fitting_rows,unlabeled],held_rows))
    scaled=preprocessor['scaler'].transform(X)
    original_labels=Y[fitting_rows].copy()
    augmented_rows=fitting_rows.copy();augmented_labels=original_labels.copy()
    R=preprocessor['ani_responsibility'];round_audit=[]
    for round_number in [1,2]:
        base=fit_ani_base(scaled[augmented_rows],augmented_labels)
        T=augmented_labels/augmented_labels.sum(axis=1,keepdims=True)
        rates=(R[augmented_rows].T@T+T.mean(axis=0))/(R[augmented_rows].sum(axis=0)[:,None]+1)
        probability=.8*predict_ani_base(base,scaled[unlabeled])+.2*(R[unlabeled]@rates)
        keep=probability.max(axis=1)>=.8;generated=decide(probability,.3)[0]
        augmented_rows=np.r_[fitting_rows,unlabeled[keep]]
        augmented_labels=np.vstack([original_labels,generated[keep]])
        round_audit.append(dict(round=round_number,generated_rows=int(keep.sum()),iterations=int(base['lr'].n_iter_.max())))
        if persist:
            FIT_AUDIT.append(dict(context=name,variant='ani_'+policy,reason='multinomial',round=round_number,
                                  fitting_rows=len(base['neighbor_targets']),verified_rows=len(fitting_rows),iterations=int(base['lr'].n_iter_.max())))
    final_base=fit_ani_base(scaled[augmented_rows],augmented_labels)
    if persist:
        FIT_AUDIT.append(dict(context=name,variant='ani_'+policy,reason='multinomial',round='final',
                              fitting_rows=len(augmented_rows),verified_rows=len(fitting_rows),iterations=int(final_base['lr'].n_iter_.max())))
        for j,reason in enumerate(REASONS):
            GENERATION_AUDIT.append(dict(context=name,variant='ani_'+policy,reason=reason,
                                         generated_positive=int(generated[keep,j].sum()),
                                         generated_negative=int((generated[keep,j]==0).sum())))
    packet=dict(family='ani',policy=policy,scaler=preprocessor['scaler'],base=final_base,native_cutoff=.3,
                features=FEATURES,reasons=REASONS,fitting_rows=fitting_rows,augmented_rows=augmented_rows,
                round_audit=round_audit,preprocessing_file=f'preprocessing/{preprocessor["name"]}.joblib')
    frame=all_frame.iloc[augmented_rows][['ID']].copy();frame['global_row']=augmented_rows
    frame['verified_fit']=np.isin(augmented_rows,fitting_rows)
    for j,reason in enumerate(REASONS):frame[reason]=augmented_labels[:,j]
    assert set(frame.ID).isdisjoint(all_frame.iloc[held_rows].ID)
    if persist:
        frame.to_csv(OUT/'datasets'/f'{name}__ani_{policy}.csv',index=False)
        pd.DataFrame(round_audit).to_csv(OUT/'datasets'/f'{name}__ani_{policy}_rounds.csv',index=False)
        pseudo=all_frame.iloc[unlabeled][['ID']].copy();pseudo['accepted_in_final_round']=keep
        for j,reason in enumerate(REASONS):
            pseudo[reason]=np.where(keep,generated[:,j],-100);pseudo[f'probability_{reason}']=probability[:,j]
        pseudo.to_csv(OUT/'datasets'/f'{name}__ani_{policy}_pseudo.csv',index=False)
    return packet,frame


def save_result(context,repeat,fold,variant,model,held_rows):
    path=OUT/'models'/f'{context}__{variant}.joblib'
    joblib.dump(model,path,compress=3)
    probability=predict_packet(model,X[held_rows])
    reloaded=joblib.load(path)
    np.testing.assert_allclose(predict_packet(reloaded,X[held_rows]),probability,rtol=1e-12,atol=1e-12)
    prediction,empty=decide(probability,model['native_cutoff'])
    frame=all_frame.iloc[held_rows][['ID']].copy();frame['global_row']=held_rows
    frame['source_group']=source_group[held_rows]
    for j,reason in enumerate(REASONS):
        frame[f'actual_{reason}']=Y[held_rows,j];frame[f'probability_{reason}']=probability[:,j]
        frame[f'predicted_{reason}']=prediction[:,j]
    frame['used_fallback']=empty
    frame['trial_f1']=2*(prediction*Y[held_rows]).sum(axis=1)/(prediction.sum(axis=1)+Y[held_rows].sum(axis=1))
    frame.to_csv(OUT/'predictions'/f'{context}__{variant}.csv',index=False)
    FOLD_RESULTS.append(dict(context=context,repeat=repeat,fold=fold,variant=variant,held_rows=len(held_rows),
                             **metrics(Y[held_rows],probability,model['native_cutoff'])))



## Exact paired fold assignments

In [3]:
SPLITS=[];balance=[]
for repeat,seed in enumerate(SEEDS):
    assignment=all_frame.iloc[known_rows][['ID']].copy();assignment['repeat']=repeat;assignment['seed']=seed;assignment['fold']=-1
    coverage=np.zeros(235,dtype=int)
    for fold,(a,b) in enumerate(KFold(5,shuffle=True,random_state=seed).split(known_rows)):
        fitting_rows=known_rows[a];held_rows=known_rows[b]
        assert len(fitting_rows)==188 and len(held_rows)==47
        assert (Y[fitting_rows].sum(axis=0)>0).all() and (Y[fitting_rows].sum(axis=0)<188).all()
        coverage[b]+=1;assignment.iloc[b,assignment.columns.get_loc('fold')]=fold
        for j,reason in enumerate(REASONS):
            balance.append(dict(repeat=repeat,fold=fold,reason=reason,fit_positive=int(Y[fitting_rows,j].sum()),held_positive=int(Y[held_rows,j].sum())))
        SPLITS.append(dict(context=f'repeat{repeat}_fold{fold}',repeat=repeat,fold=fold,fitting_rows=fitting_rows,held_rows=held_rows))
    np.testing.assert_array_equal(coverage,np.ones(235,dtype=int))
    assignment.to_csv(OUT/'splits'/f'repeat{repeat}.csv',index=False)
pd.DataFrame(balance).to_csv(OUT/'splits/label_balance.csv',index=False)
print('All 40 exact paired folds saved. Every labeled trial is held out once per repeat.')



All 40 exact paired folds saved. Every labeled trial is held out once per repeat.


## Reproduce both original validation endpoints

In [4]:
GLOBAL=prepare_features('transductive_all1561',ALL_ROWS)
with threadpool_limits(limits=1):
    ani_control,ani_control_frame=fit_ani('validation_control','transductive',original_known,valid_rows,GLOBAL,persist=False)
    probability=predict_packet(ani_control,X[valid_rows])
    saved_probability=predict_ani_base(ANI_COMPONENT['base'],ANI_COMPONENT['scaler'].transform(X[valid_rows]))
    np.testing.assert_allclose(probability,saved_probability,rtol=1e-8,atol=1e-10)
    assert len(ani_control_frame)-157==985
    np.testing.assert_allclose(metrics(Y[valid_rows],probability,.3)['sample_f1'],.8931623931623932,atol=1e-14)
    db_prep=prepare_features('validation_dbscan_control',np.arange(len(train)))
    db_control,db_frame=fit_dbscan('validation_control','train_only',original_known,valid_rows,db_prep,persist=False)
    old_probability=np.column_stack([e.predict_proba(OLD_MODEL['scaler'].transform(X[valid_rows]))[:,1] for e in OLD_MODEL['estimators']])
    np.testing.assert_allclose(predict_packet(db_control,X[valid_rows]),old_probability,rtol=1e-8,atol=1e-10)
    fixed=pd.read_csv(SOURCE/'datasets/dbscan_m5_q25.csv',float_precision='round_trip')
    np.testing.assert_array_equal(db_frame[REASONS],fixed[REASONS])
    np.testing.assert_allclose(db_frame[[f'weight_{r}' for r in REASONS]],fixed[[f'weight_{r}' for r in REASONS]],atol=1e-12)
joblib.dump(ani_control,OUT/'controls/ani_validation_control.joblib',compress=3)
joblib.dump(db_control,OUT/'controls/dbscan_validation_control.joblib',compress=3)
ani_control_frame.to_csv(OUT/'controls/ani_validation_control_targets.csv',index=False)
db_frame.to_csv(OUT/'controls/dbscan_validation_control_targets_weights.csv',index=False)
print('Verified both native validation endpoints: Ani 0.893162 (985 generated rows); DBSCAN 0.910256 (exact saved targets/weights).')



Verified both native validation endpoints: Ani 0.893162 (985 generated rows); DBSCAN 0.910256 (exact saved targets/weights).


## Run a repeat with all four variants

In [5]:
def run_repeat(repeat):
    start=time.monotonic()
    for spec in [s for s in SPLITS if s['repeat']==repeat]:
        name=spec['context'];fitting_rows=spec['fitting_rows'];held_rows=spec['held_rows']
        strict_pool=np.sort(np.r_[fitting_rows,original_unlabeled])
        assert len(strict_pool)==1202 and not len(np.intersect1d(strict_pool,np.r_[held_rows,test_rows]))
        strict=prepare_features(name+'_train_only',strict_pool)
        roles=all_frame[['ID']].copy();roles['role']='unlabeled_training'
        roles.loc[test_rows,'role']='unlabeled_test';roles.loc[fitting_rows,'role']='verified_fit';roles.loc[held_rows,'role']='held_out'
        roles.to_csv(OUT/'splits'/f'{name}_roles.csv',index=False)
        for variant in VARIANTS:
            family='ani' if variant.startswith('ani_') else 'dbscan'
            policy='transductive' if variant.endswith('transductive') else 'train_only'
            prep=GLOBAL if policy=='transductive' else strict
            with threadpool_limits(limits=1):
                function=fit_ani if family=='ani' else fit_dbscan
                model,frame=function(name,policy,fitting_rows,held_rows,prep)
                save_result(name,repeat,spec['fold'],variant,model,held_rows)
            assert set(frame.ID).isdisjoint(all_frame.iloc[held_rows].ID)
            if policy=='train_only':assert set(frame.ID).isdisjoint(test.ID)
            ROLE_AUDIT.append(dict(context=name,variant=variant,verified_fit_rows=188,held_rows=47,
                                   classifier_held_label_overlap=0,pseudo_pool_held_overlap=0,
                                   feature_fit_rows=len(prep['feature_rows']),test_features_used=policy=='transductive',
                                   augmented_or_pool_rows=len(frame)))
        pd.DataFrame(FOLD_RESULTS).to_csv(OUT/'fold_scores.csv',index=False)
        pd.DataFrame(FIT_AUDIT).to_csv(OUT/'fit_audit.csv',index=False)
        pd.DataFrame(GENERATION_AUDIT).to_csv(OUT/'generated_labels.csv',index=False)
        pd.DataFrame(ROLE_AUDIT).to_csv(OUT/'role_audit.csv',index=False)
        print(name,{row['variant']:round(row['sample_f1'],5) for row in FOLD_RESULTS[-4:]},flush=True)
    print(f'Repeat {repeat} complete in {time.monotonic()-start:.1f}s.',flush=True)



## Paired CV repeat 1 of 8

In [6]:
run_repeat(0)




repeat0_fold0 {'dbscan_train_only': 0.85106, 'ani_transductive': 0.85816, 'ani_train_only': 0.85106, 'dbscan_transductive': 0.85106}


repeat0_fold1 {'dbscan_train_only': 0.88652, 'ani_transductive': 0.91489, 'ani_train_only': 0.90071, 'dbscan_transductive': 0.87943}


repeat0_fold2 {'dbscan_train_only': 0.89362, 'ani_transductive': 0.95035, 'ani_train_only': 0.95035, 'dbscan_transductive': 0.89362}


repeat0_fold3 {'dbscan_train_only': 0.9078, 'ani_transductive': 0.92908, 'ani_train_only': 0.92199, 'dbscan_transductive': 0.9078}


repeat0_fold4 {'dbscan_train_only': 0.92908, 'ani_transductive': 0.96454, 'ani_train_only': 0.95745, 'dbscan_transductive': 0.92908}


Repeat 0 complete in 2.4s.


## Paired CV repeat 2 of 8

In [7]:
run_repeat(1)




repeat1_fold0 {'dbscan_train_only': 0.87943, 'ani_transductive': 0.92553, 'ani_train_only': 0.93617, 'dbscan_transductive': 0.87943}


repeat1_fold1 {'dbscan_train_only': 0.85816, 'ani_transductive': 0.92199, 'ani_train_only': 0.91489, 'dbscan_transductive': 0.85816}


repeat1_fold2 {'dbscan_train_only': 0.88936, 'ani_transductive': 0.92908, 'ani_train_only': 0.92199, 'dbscan_transductive': 0.88936}


repeat1_fold3 {'dbscan_train_only': 0.92199, 'ani_transductive': 0.95035, 'ani_train_only': 0.95035, 'dbscan_transductive': 0.92199}


repeat1_fold4 {'dbscan_train_only': 0.90071, 'ani_transductive': 0.85106, 'ani_train_only': 0.86525, 'dbscan_transductive': 0.90071}


Repeat 1 complete in 2.4s.


## Paired CV repeat 3 of 8

In [8]:
run_repeat(2)




repeat2_fold0 {'dbscan_train_only': 0.93617, 'ani_transductive': 0.88652, 'ani_train_only': 0.87943, 'dbscan_transductive': 0.93617}


repeat2_fold1 {'dbscan_train_only': 0.84752, 'ani_transductive': 0.92908, 'ani_train_only': 0.91489, 'dbscan_transductive': 0.84043}


repeat2_fold2 {'dbscan_train_only': 0.8227, 'ani_transductive': 0.84752, 'ani_train_only': 0.85461, 'dbscan_transductive': 0.8227}


repeat2_fold3 {'dbscan_train_only': 0.91773, 'ani_transductive': 0.97872, 'ani_train_only': 0.95745, 'dbscan_transductive': 0.91773}


repeat2_fold4 {'dbscan_train_only': 0.89362, 'ani_transductive': 0.88652, 'ani_train_only': 0.88652, 'dbscan_transductive': 0.89362}


Repeat 2 complete in 2.2s.


## Paired CV repeat 4 of 8

In [9]:
run_repeat(3)




repeat3_fold0 {'dbscan_train_only': 0.88298, 'ani_transductive': 0.91135, 'ani_train_only': 0.89007, 'dbscan_transductive': 0.88298}


repeat3_fold1 {'dbscan_train_only': 0.91489, 'ani_transductive': 0.92908, 'ani_train_only': 0.92908, 'dbscan_transductive': 0.9078}


repeat3_fold2 {'dbscan_train_only': 0.86525, 'ani_transductive': 0.86525, 'ani_train_only': 0.84397, 'dbscan_transductive': 0.86525}


repeat3_fold3 {'dbscan_train_only': 0.87589, 'ani_transductive': 0.90071, 'ani_train_only': 0.9078, 'dbscan_transductive': 0.87589}


repeat3_fold4 {'dbscan_train_only': 0.89716, 'ani_transductive': 0.93617, 'ani_train_only': 0.93617, 'dbscan_transductive': 0.89716}


Repeat 3 complete in 2.5s.


## Paired CV repeat 5 of 8

In [10]:
run_repeat(4)




repeat4_fold0 {'dbscan_train_only': 0.9078, 'ani_transductive': 0.95745, 'ani_train_only': 0.9539, 'dbscan_transductive': 0.9078}


repeat4_fold1 {'dbscan_train_only': 0.9078, 'ani_transductive': 0.88652, 'ani_train_only': 0.87943, 'dbscan_transductive': 0.9078}


repeat4_fold2 {'dbscan_train_only': 0.85106, 'ani_transductive': 0.90071, 'ani_train_only': 0.90071, 'dbscan_transductive': 0.85106}


repeat4_fold3 {'dbscan_train_only': 0.89645, 'ani_transductive': 0.92908, 'ani_train_only': 0.91489, 'dbscan_transductive': 0.89645}


repeat4_fold4 {'dbscan_train_only': 0.90071, 'ani_transductive': 0.91489, 'ani_train_only': 0.91135, 'dbscan_transductive': 0.90071}


Repeat 4 complete in 2.6s.


## Paired CV repeat 6 of 8

In [11]:
run_repeat(5)




repeat5_fold0 {'dbscan_train_only': 0.88652, 'ani_transductive': 0.90071, 'ani_train_only': 0.88652, 'dbscan_transductive': 0.87943}


repeat5_fold1 {'dbscan_train_only': 0.85106, 'ani_transductive': 0.89362, 'ani_train_only': 0.88652, 'dbscan_transductive': 0.85106}


repeat5_fold2 {'dbscan_train_only': 0.88652, 'ani_transductive': 0.91489, 'ani_train_only': 0.91489, 'dbscan_transductive': 0.88652}


repeat5_fold3 {'dbscan_train_only': 0.93617, 'ani_transductive': 0.97163, 'ani_train_only': 0.95035, 'dbscan_transductive': 0.93617}


repeat5_fold4 {'dbscan_train_only': 0.88652, 'ani_transductive': 0.9078, 'ani_train_only': 0.90071, 'dbscan_transductive': 0.88652}


Repeat 5 complete in 2.8s.


## Paired CV repeat 7 of 8

In [12]:
run_repeat(6)




repeat6_fold0 {'dbscan_train_only': 0.88652, 'ani_transductive': 0.9078, 'ani_train_only': 0.90071, 'dbscan_transductive': 0.88652}


repeat6_fold1 {'dbscan_train_only': 0.88227, 'ani_transductive': 0.90071, 'ani_train_only': 0.92199, 'dbscan_transductive': 0.88227}


repeat6_fold2 {'dbscan_train_only': 0.83688, 'ani_transductive': 0.90071, 'ani_train_only': 0.91489, 'dbscan_transductive': 0.83688}


repeat6_fold3 {'dbscan_train_only': 0.92199, 'ani_transductive': 0.92908, 'ani_train_only': 0.91489, 'dbscan_transductive': 0.92199}


repeat6_fold4 {'dbscan_train_only': 0.9078, 'ani_transductive': 0.92199, 'ani_train_only': 0.88652, 'dbscan_transductive': 0.9078}


Repeat 6 complete in 2.4s.


## Paired CV repeat 8 of 8

In [13]:
run_repeat(7)




repeat7_fold0 {'dbscan_train_only': 0.85816, 'ani_transductive': 0.90071, 'ani_train_only': 0.89362, 'dbscan_transductive': 0.85816}


repeat7_fold1 {'dbscan_train_only': 0.89362, 'ani_transductive': 0.90071, 'ani_train_only': 0.90071, 'dbscan_transductive': 0.89362}


repeat7_fold2 {'dbscan_train_only': 0.93617, 'ani_transductive': 0.97163, 'ani_train_only': 0.95035, 'dbscan_transductive': 0.93617}


repeat7_fold3 {'dbscan_train_only': 0.84752, 'ani_transductive': 0.87234, 'ani_train_only': 0.87234, 'dbscan_transductive': 0.84752}


repeat7_fold4 {'dbscan_train_only': 0.87943, 'ani_transductive': 0.92199, 'ani_train_only': 0.92908, 'dbscan_transductive': 0.87943}


Repeat 7 complete in 2.5s.


## Pool out-of-fold scores and compare paired results

In [14]:
folds=pd.DataFrame(FOLD_RESULTS)
assert len(folds)==160
repeat_results=[];subgroup_results=[]
for repeat in range(8):
    for variant in VARIANTS:
        frames=[pd.read_csv(OUT/'predictions'/f'repeat{repeat}_fold{fold}__{variant}.csv',float_precision='round_trip') for fold in range(5)]
        merged=pd.concat(frames,ignore_index=True)
        assert len(merged)==235 and set(merged.ID)==set(all_frame.iloc[known_rows].ID)
        probability=merged[[f'probability_{r}' for r in REASONS]].to_numpy()
        actual=merged[[f'actual_{r}' for r in REASONS]].to_numpy(dtype=int)
        cutoff=.3 if variant.startswith('ani_') else .5
        repeat_results.append(dict(repeat=repeat,variant=variant,**metrics(actual,probability,cutoff)))
        for group in ['original_training','former_validation']:
            mask=merged.source_group.to_numpy()==group
            subgroup_results.append(dict(repeat=repeat,variant=variant,source_group=group,
                                          trials=int(mask.sum()),**metrics(actual[mask],probability[mask],cutoff)))
repeats=pd.DataFrame(repeat_results);subgroups=pd.DataFrame(subgroup_results)
repeats.to_csv(OUT/'repeat_scores.csv',index=False);subgroups.to_csv(OUT/'subgroup_scores.csv',index=False)
summary=[]
for variant in VARIANTS:
    r=repeats[repeats.variant==variant]
    summary.append(dict(variant=variant,cv_sample_f1=float(r.sample_f1.mean()),repeat_sd=float(r.sample_f1.std(ddof=1)),
                        cv_macro_f1=float(r.macro_f1.mean()),minimum_repeat=float(r.sample_f1.min()),maximum_repeat=float(r.sample_f1.max()),
                        **{f'cv_f1_{reason}':float(r[f'f1_{reason}'].mean()) for reason in REASONS}))
comparison=pd.DataFrame(summary)
comparison.to_csv(OUT/'comparison.csv',index=False)
paired=[];paired_repeats=[]
for name,(db_variant,ani_variant) in COMPARISONS.items():
    d=folds[folds.variant==db_variant].set_index('context').sample_f1
    a=folds[folds.variant==ani_variant].set_index('context').sample_f1
    delta=a-d
    rd=repeats[repeats.variant==db_variant].set_index('repeat').sample_f1
    ra=repeats[repeats.variant==ani_variant].set_index('repeat').sample_f1
    dr=ra-rd
    for repeat,value in dr.items():paired_repeats.append(dict(comparison=name,repeat=repeat,ani_minus_dbscan=float(value)))
    paired.append(dict(comparison=name,dbscan_variant=db_variant,ani_variant=ani_variant,
                       dbscan_cv_sample_f1=float(rd.mean()),ani_cv_sample_f1=float(ra.mean()),
                       ani_minus_dbscan=float(dr.mean()),ani_fold_wins=int((delta>1e-12).sum()),
                       fold_ties=int((abs(delta)<=1e-12).sum()),ani_fold_losses=int((delta< -1e-12).sum()),
                       ani_repeat_wins=int((dr>1e-12).sum()),repeat_ties=int((abs(dr)<=1e-12).sum()),
                       ani_repeat_losses=int((dr< -1e-12).sum())))
paired=pd.DataFrame(paired);paired.to_csv(OUT/'paired_comparison.csv',index=False)
pd.DataFrame(paired_repeats).to_csv(OUT/'paired_repeat_differences.csv',index=False)
display(comparison)
display(paired)
display(subgroups.groupby(['variant','source_group'])[['sample_f1','macro_f1']].mean())



,variant,cv_sample_f1,repeat_sd,cv_macro_f1,minimum_repeat,maximum_repeat,cv_f1_Safety,cv_f1_Funding,cv_f1_Efficacy,cv_f1_Recruitment,cv_f1_Other
0,dbscan_train_only,0.888316,0.003882,0.807876,0.882979,0.893617,0.868164,0.753986,0.853972,0.956423,0.606836
1,ani_transductive,0.914273,0.005628,0.838763,0.905674,0.923404,0.933488,0.816733,0.869190,0.973279,0.601125
2,ani_train_only,0.908865,0.006631,0.822043,0.898582,0.917730,0.907372,0.760623,0.869526,0.972481,0.600213
3,dbscan_transductive,0.887606,0.003930,0.806349,0.882128,0.892766,0.868164,0.753986,0.846336,0.956423,0.606836


,comparison,dbscan_variant,ani_variant,dbscan_cv_sample_f1,ani_cv_sample_f1,ani_minus_dbscan,ani_fold_wins,fold_ties,ani_fold_losses,ani_repeat_wins,repeat_ties,ani_repeat_losses
0,native_recipes,dbscan_train_only,ani_transductive,0.888316,0.914273,0.025957,35,1,4,8,0,0
1,shared_train_only,dbscan_train_only,ani_train_only,0.888316,0.908865,0.020550,31,2,7,8,0,0
2,shared_transductive,dbscan_transductive,ani_transductive,0.887606,0.914273,0.026667,35,1,4,8,0,0


sample_f1  macro_f1
variant             source_group                          
ani_train_only      former_validation   0.881677  0.752447
                    original_training   0.922373  0.857053
ani_transductive    former_validation   0.882212  0.755837
                    original_training   0.930202  0.879056
dbscan_train_only   former_validation   0.871528  0.763085
                    original_training   0.896656  0.830496
dbscan_transductive former_validation   0.870459  0.761106
                    original_training   0.896125  0.829189

## Independent source, dataset, and artifact verification

In [15]:
# Direct source-code check for one CV fold: use Ani's original functions, not
# just a second call to our wrapper. No output contains the source notebook's images.
snapshot=json.loads(ANI_SOURCE.read_text())
source_functions='\n\n'.join(''.join(c['source']) for c in snapshot['cells'] if c['cell_type']=='code' and 'def final_model' in ''.join(c['source']))
namespace=dict(np=np,normalize=normalize,LogisticRegression=LogisticRegression,
               gmm=GLOBAL['gmm'],pca30=GLOBAL['pca'],EMB=FEATURES)
exec(compile(source_functions,'ani_original_model_functions','exec'),namespace)
first=SPLITS[0];scaled=GLOBAL['scaler'].transform(X)
with threadpool_limits(limits=1):
    original_probability=namespace['final_model'](scaled[first['fitting_rows']],Y[first['fitting_rows']],
        scaled[first['held_rows']],scaled[np.r_[original_unlabeled,test_rows]])
packet=joblib.load(OUT/'models/repeat0_fold0__ani_transductive.joblib')
np.testing.assert_allclose(original_probability,predict_packet(packet,X[first['held_rows']]),rtol=1e-8,atol=1e-10)

checks=[]
for row in folds.itertuples():
    frame=pd.read_csv(OUT/'predictions'/f'{row.context}__{row.variant}.csv',float_precision='round_trip')
    actual=frame[[f'actual_{r}' for r in REASONS]].to_numpy(dtype=int)
    probability=frame[[f'probability_{r}' for r in REASONS]].to_numpy()
    cutoff=.3 if row.variant.startswith('ani_') else .5
    rescored=metrics(actual,probability,cutoff)
    np.testing.assert_allclose([rescored['sample_f1'],rescored['macro_f1']],[row.sample_f1,row.macro_f1],atol=1e-14)
    np.testing.assert_array_equal(decide(probability,cutoff)[0],frame[[f'predicted_{r}' for r in REASONS]])
    checks.append(dict(context=row.context,variant=row.variant,passed=True))
pd.DataFrame(checks).to_csv(OUT/'disk_score_checks.csv',index=False)

# Independently replay final fitting weights/targets for all four variants in
# the first context from exported datasets and shared raw features.
replay_checks=[]
for variant in VARIANTS:
    packet=joblib.load(OUT/'models'/f'repeat0_fold0__{variant}.joblib')
    frame=pd.read_csv(OUT/'datasets'/f'repeat0_fold0__{variant}.csv',float_precision='round_trip')
    rows=frame.global_row.to_numpy(dtype=int)
    assert frame.ID.tolist()==all_frame.iloc[rows].ID.tolist()
    assert set(frame.ID).isdisjoint(all_frame.iloc[first['held_rows']].ID)
    design=packet['scaler'].transform(X[rows])
    with threadpool_limits(limits=1):
        if packet['family']=='ani':
            replay={**packet,'base':fit_ani_base(design,frame[REASONS].to_numpy(dtype=int))}
        else:
            replay={**packet,'estimators':[]}
            for j,reason in enumerate(REASONS):
                weight=frame[f'normalized_weight_{reason}'].to_numpy();included=weight>0
                estimator=clone(packet['estimators'][j])
                with warnings.catch_warnings():
                    warnings.simplefilter('error',ConvergenceWarning)
                    estimator.fit(design[included],frame[reason].to_numpy(dtype=int)[included],sample_weight=weight[included])
                replay['estimators'].append(estimator)
    np.testing.assert_allclose(predict_packet(replay,X[first['held_rows']]),predict_packet(packet,X[first['held_rows']]),rtol=1e-8,atol=1e-10)
    replay_checks.append(dict(context='repeat0_fold0',variant=variant,passed=True))
pd.DataFrame(replay_checks).to_csv(OUT/'independent_replays.csv',index=False)
for relative,digest in PROTECTED.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest
assert len(repeats)==32 and len(ROLE_AUDIT)==160 and len(checks)==160
assert all(r['classifier_held_label_overlap']==r['pseudo_pool_held_overlap']==0 for r in ROLE_AUDIT)
print('Verified 160 saved model reloads and disk scores, four independent dataset refits, one direct Ani-source CV check, exact fold isolation, and protected hashes.')



Verified 160 saved model reloads and disk scores, four independent dataset refits, one direct Ani-source CV check, exact fold isolation, and protected hashes.


## Save scores, audit details, and interpretation

In [16]:
settings=dict(protocol=PROTOCOL,fold_evaluations=160,repeat_evaluations=32,
              primary_fit_count=len(FIT_AUDIT),all_saved_models_reloaded=True,independent_refits=4,
              ani_source_cv_reproduction_checked=True,models_retrained_for_kaggle=False,
              no_new_submission=True,protocol_selected_by_kaggle_agreement=False,
              dataset_sha256=hashlib.sha256((OUT/'datasets/shared_raw_features_labels.csv').read_bytes()).hexdigest(),
              versions=dict(python=platform.python_version(),sklearn=sklearn.__version__,numpy=np.__version__,pandas=pd.__version__))
(OUT/'settings.json').write_text(json.dumps(settings,indent=2)+'\n')
lines=['# Paired CV: Ani and DBSCAN recipes','',
       'Fixed before fitting: eight repeats of five-fold CV over the same 235 verified labels, '
       '188 fitting labels / 47 held out per model, seeds 100–107 and KFold ordering matching Ani’s original CV. '
       'Both recipes use the same folds and primary sample-F1 metric. No parameter tuning or new submission.','',
       '## Four variants','',
       '| Variant | Mean sample F1 | Repeat SD | Mean macro F1 |','|---|---:|---:|---:|']
for row in comparison.itertuples():
    lines.append(f'| {row.variant} | {row.cv_sample_f1:.6f} | {row.repeat_sd:.6f} | {row.cv_macro_f1:.6f} |')
lines+=['','## Paired comparisons','','| Comparison | DBSCAN F1 | Ani F1 | Ani minus DBSCAN | Ani fold wins / ties / losses | Ani repeat wins / ties / losses |','|---|---:|---:|---:|---|---|']
for row in paired.itertuples():
    lines.append(f'| {row.comparison} | {row.dbscan_cv_sample_f1:.6f} | {row.ani_cv_sample_f1:.6f} | {row.ani_minus_dbscan:+.6f} | '
                 f'{row.ani_fold_wins}/{row.fold_ties}/{row.ani_fold_losses} | {row.ani_repeat_wins}/{row.repeat_ties}/{row.ani_repeat_losses} |')
lines+=['','Primary **native_recipes** preserves the original feature policies: DBSCAN training-only, Ani transductive. '
        'The two shared-policy comparisons deliberately change feature access to separate it from recipe architecture. '
        'All policies and comparisons were declared before scores; none was chosen because it resembles Kaggle.','',
        '## Label and feature access','',
        'Every fold withholds the same 47 known labels from all supervised fitting, cluster-label mapping, and pseudo-label targets. '
        'Held-out known rows never enter augmentation, even under transductive preprocessing. Training-only feature fitting '
        'uses 188 verified + 1,014 originally unlabeled training trials and excludes held-out/test features entirely. '
        'Transductive feature-only scaler/PCA/GMM or DBSCAN use all 1,561 embeddings, never held-out/test labels; '
        'both recipes may augment from the same 1,014 unlabeled training + 312 test rows. '
        'DBSCAN classifier scaling stays verified-fitting-only, while Ani scaling uses the permitted feature pool, preserving recipe choices.','',
        'DBSCAN retains C=1000, LBFGS/tol=1e-4/max_iter=5000, min_samples=5, q25 radius, conservative support/confidence/weight rules, '
        'cutoff 0.5 and fallback. Ani retains C=0.3 balanced multinomial LR + cosine kNN10 (70/30), '
        'PCA30/GMM30-diagonal, 80/20 teacher/GMM pseudo-label scores, confidence 0.8, two rounds, cutoff 0.3 and top-label inclusion.','',
        '## What the scores can establish','',
        'Mean sample F1 pools all 235 out-of-fold trial predictions per repeat, then averages eight repeats. '
        'Because every fold has 47 trials, it also equals the mean of the 40 fold sample-F1 values. '
        'Macro F1 is pooled within repeat before averaging; it may differ from Ani’s original mean fold macro F1. '
        'Repeats share trials/training examples: SD and paired win counts are descriptive, not independent-test significance claims.','',
        'The old 78-trial validation and previous 157-label three-fold CV are separate settings and should not be compared numerically '
        'as if identical to this 235-label five-fold evaluation. Its recipes have already been selected using these data, '
        'so this is a consistent exploratory benchmark, not a fresh independent test. '
        'Kaggle trials are later than the labeled pool; random folds cannot reproduce that shift. '
        'Agreement with the known Kaggle ordering for two recipes is useful evidence, not proof that future rankings will transfer.','',
        'Known user-reported Kaggle scores: Ani after 235-label refit **0.89423**; DBSCAN original 157-label fit **0.87500**; '
        'DBSCAN after 235-label refit **0.86452**. This CV evaluates fits using 188 verified labels and therefore '
        'does not reproduce either complete final submission fit.','',
        '## Reproduction and artifacts','',
        'Notebook: part1/paired_recipe_cv.ipynb. protocol.json/PROTOCOL.md fix all choices before fitting. '
        'splits/ preserves all fold assignments and row roles. datasets/ saves shared raw features/visible labels '
        'and every fold’s exact DBSCAN targets/raw/normalized weights or Ani augmented targets/self-training audits. '
        'preprocessing/ saves fold-local or feature-only global preprocessing and cluster models. models/ and '
        'predictions/ preserve all 160 fitted variants and held-out probabilities/labels. comparison.csv, '
        'paired_comparison.csv, repeat_scores.csv, subgroup_scores.csv and fit/role/generation audits support the tables.','',
        'Both native validation endpoints reproduce their original saved probabilities and scores. The first native Ani CV fit '
        'also matches direct execution of Ani’s original model functions. All 160 model packets reload and independently '
        'rescore from persisted predictions; all four first-context variants independently refit from exported targets/weights. '
        'Original source models, CSVs, and data keep their hashes. No Kaggle upload, new submission CSV, commit, push, or tag was made.']
(OUT/'SUMMARY.md').write_text('\n'.join(lines)+'\n')
print('Saved paired comparison:',OUT/'SUMMARY.md')


Saved paired comparison: /Users/michael/Documents/School/Cornell/AppliedMachineLearning/AML-Midterm/part1/experiments/paired_recipe_cv/SUMMARY.md


## Observed results at this checkpoint

Ani’s original recipe averages **0.914273** sample F1, compared with **0.888316** for the original DBSCAN recipe. Ani wins **35/40** folds, ties **1**, and wins the mean score in **all eight repeats**. The direction agrees with the known Kaggle ordering.

With both recipes restricted to training-only feature access, Ani remains ahead: **0.908865** versus **0.888316**. With transductive access available to both, the scores are **0.914273** versus **0.887606**. The advantage is not explained solely by extra feature access in these controls.

Use Ani as the fixed reference for subsequent recipes, preserving these exact folds and the declared feature-access policy. Agreement with two known submission recipes is supporting evidence; it does not guarantee that future CV gains will transfer to later Kaggle trials. See BENCHMARK.md for the comparison contract.
